# StyleSense Recommendation Pipeline

This notebook trains and evaluates one scikit-learn pipeline for mixed numerical, categorical, and review-text data. Missing values are imputed inside the pipeline, and GridSearchCV tunes the estimator using training data only.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from data_generation import load_or_create_dataset
from model import RecommendationModel

## Load and inspect the data

In [ ]:
project_dir = Path.cwd()
df = load_or_create_dataset(project_dir / 'data', n_samples=1000)
display(df.head())
display(df['recommend'].value_counts(normalize=True).rename('recommendation_rate'))

## Split raw records before fitting

The test set remains untouched until final evaluation. No preprocessing is fitted before the split.

In [ ]:
X = df.drop(columns='recommend')
y = df['recommend']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'train rows: {len(X_train)}, test rows: {len(X_test)}')

## Train the end-to-end tuned pipeline

In [ ]:
recommendation_model = RecommendationModel(
    model_type='random_forest', random_state=42
)
recommendation_model.train(X_train, y_train)
print('Best parameters:', recommendation_model.best_params_)
print('Best cross-validation F1:', recommendation_model.best_cv_score_)

## Evaluate on held-out records

In [ ]:
test_predictions = recommendation_model.predict(X_test)
test_probabilities = recommendation_model.predict_proba(X_test)[:, 1]
print('Accuracy:', accuracy_score(y_test, test_predictions))
print('F1:', f1_score(y_test, test_predictions))
print('ROC-AUC:', roc_auc_score(y_test, test_probabilities))
print(classification_report(y_test, test_predictions, zero_division=0))

## Predict raw records

The fitted object owns both preprocessing and the estimator. New records can be passed directly to `predict`, including records with missing values.

In [ ]:
new_review = pd.DataFrame([{
    'age': 28,
    'category': 'Dresses',
    'price_range': 'Mid-Range',
    'rating': 5,
    'review_text': 'Amazing quality and perfect fit. I would recommend it.'
}])
print('Prediction:', recommendation_model.predict(new_review)[0])
print('Recommendation probability:', recommendation_model.predict_proba(new_review)[0, 1])